# 04. Escenarios condicionados al estado del mercado

## 1. ¿Cómo incorporar el estado del mercado a nuestros escenarios?

En la notebook 03 construimos escenarios seleccionando observaciones históricas completas. Primero asignamos la misma probabilidad a todas las sesiones; después dimos mayor peso a las recientes.

Sin embargo, la antigüedad de una observación no describe las condiciones que precedieron a sus movimientos. Una sesión reciente puede haber ocurrido en un entorno distinto del que queremos representar.

Esto plantea una nueva pregunta: ¿podemos modificar las probabilidades históricas para dar relevancia a las condiciones del mercado, además de a la recencia?

Para investigarlo incorporamos el VIX como una primera variable de estado. Cada vector histórico de 125 retornos tiene asociado un nivel previo del índice, siguiendo una convención temporal explícita.

Los retornos observados no cambian. Lo que investigaremos es cómo asignarles probabilidades distintas según el estado que queremos representar. Que podamos construir esa distribución no demuestra todavía que produzca mejores pronósticos.

### ¿Qué observaciones podemos utilizar?

$A$ original contiene 2,204 observaciones conjuntas de 125 instrumentos. La tabla de auditoría conserva esas 2,204 filas: hay 2,203 filas alineadas y una no alineable por falta de antecedente dentro del calendario publicado. El subconjunto de trabajo conserva exactamente los retornos originales y el orden de instrumentos. Las comparaciones posteriores dentro de esta notebook utilizarán la misma muestra alineada para todos los métodos.

Para un instrumento $j$, el retorno logarítmico fechado en la sesión $x$ es:

$$\ell[x,j] = \log\left(\frac{P[x,j]}{P[x-1,j]}\right).$$

Aquí $x-1$ y $x-2$ son sesiones del calendario de acciones del corte publicado, no días calendario ni posiciones del calendario propio de VIX. El retorno fechado en $x$ va del cierre de $x-1$ al cierre de $x$; su VIX asociado corresponde a $x-2$. La regla deja una sesión completa de margen para publicación.

Alpha Vantage comunicó disponibilidad habitual para las 22:00 US/Eastern. Es una convención de disponibilidad, no una verificación histórica point-in-time ni garantía de ausencia de revisiones.

## 2. Carga y validación offline

Esta notebook es autónoma: no ejecuta las notebooks 02 o 03, no usa API ni credenciales y no persiste matrices nuevas. Recupera los log-retornos con el mismo procedimiento validado en la notebook 02 y verifica las referencias de integridad disponibles. Los hashes de los Parquet matriciales registrados durante la alineación fueron calculados en esa ejecución; el manifiesto sellado del corte no aporta hashes históricos de referencia para esos dos Parquet.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import site
import sys
from datetime import date
from pathlib import Path

import numpy as np

def find_project_root() -> Path:
    def is_project_root(candidate: Path) -> bool:
        return ((candidate / 'config' / 'universe.csv').is_file()
                and (candidate / 'src' / 'scenario_portfolio_lab').is_dir())

    for origin in (Path.cwd(), *Path.cwd().parents):
        for candidate in (origin, origin / 'scenario-portfolio-lab', origin / 'data-science-portfolio' / 'scenario-portfolio-lab'):
            if is_project_root(candidate):
                return candidate
    raise RuntimeError(
        'Repository files config/universe.csv and src/scenario_portfolio_lab were not found. '
        'Open this notebook from the complete scenario-portfolio-lab repository (or its data-science-portfolio parent). '
        'Do not run it from an exported standalone .ipynb file.'
    )


ROOT = find_project_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# MyST/Jupyter kernels may differ from the project virtual environment.
# Reuse already installed local dependencies when this Windows project .venv exists; never install packages here.
LOCAL_SITE_PACKAGES = ROOT / '.venv' / 'Lib' / 'site-packages'
if LOCAL_SITE_PACKAGES.is_dir():
    site.addsitedir(str(LOCAL_SITE_PACKAGES))

try:
    import exchange_calendars as xcals
    import polars as pl
except ModuleNotFoundError as exc:
    raise RuntimeError(
        'Missing project dependencies. Select the project .venv Python 3.13.2 kernel or install requirements.txt '
        'into the active kernel environment before running this notebook.'
    ) from exc

RUN_ROOT = ROOT / 'data' / 'published' / 'cut_2026-10-09_run_2026-10-11_full_window_final'
GOLD_ROOT = RUN_ROOT / 'gold'
PRICE_MATRIX_PATH = GOLD_ROOT / 'adjusted_close_matrix.parquet'
RETURN_MATRIX_PATH = GOLD_ROOT / 'eligible_simple_total_return_matrix.parquet'
RUN_MANIFEST_PATH = RUN_ROOT / 'run_manifest.json'
ELIGIBLE_CATALOG_PATH = RUN_ROOT / 'reports' / 'eligible_universe.csv'
ALIGNMENT_ROOT = ROOT / 'data' / 'published' / 'vix_state_alignment_2026-10-31'
ALIGNMENT_TABLE_PATH = ALIGNMENT_ROOT / 'vix_return_alignment.parquet'
ALIGNMENT_MANIFEST_PATH = ALIGNMENT_ROOT / 'alignment_manifest.json'
ALIGNMENT_REPORT_PATH = ALIGNMENT_ROOT / 'control_report.json'
METHODOLOGY_PATH = ALIGNMENT_ROOT / 'methodology.json'
VIX_ROOT = ROOT / 'data' / 'raw' / 'vix_index_access_2026-10-29'
VIX_RAW_PATH = VIX_ROOT / 'raw' / 'VIX_INDEX_DATA_daily.json'
VIX_CAPTURE_MANIFEST_PATH = VIX_ROOT / 'manifest.json'

for required in (PRICE_MATRIX_PATH, RETURN_MATRIX_PATH, RUN_MANIFEST_PATH, ELIGIBLE_CATALOG_PATH, ALIGNMENT_TABLE_PATH, ALIGNMENT_MANIFEST_PATH, ALIGNMENT_REPORT_PATH, METHODOLOGY_PATH, VIX_RAW_PATH, VIX_CAPTURE_MANIFEST_PATH):
    if not required.is_file():
        raise FileNotFoundError(f'Required local artifact is missing: {required.relative_to(ROOT)}')

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


In [2]:
run_manifest = json.loads(RUN_MANIFEST_PATH.read_text(encoding='utf-8'))
alignment_manifest = json.loads(ALIGNMENT_MANIFEST_PATH.read_text(encoding='utf-8'))
alignment_report = json.loads(ALIGNMENT_REPORT_PATH.read_text(encoding='utf-8'))
methodology = json.loads(METHODOLOGY_PATH.read_text(encoding='utf-8'))
vix_capture_manifest = json.loads(VIX_CAPTURE_MANIFEST_PATH.read_text(encoding='utf-8'))
approved_period = run_manifest['approved_period']
assert approved_period == {'start': '2018-01-02', 'end': '2026-10-09'}
start, end = date.fromisoformat(approved_period['start']), date.fromisoformat(approved_period['end'])

# Historical raw references exist for the 138 captured instruments.
raw_hash_checks = []
for symbol, reference in run_manifest['raw'].items():
    raw_path = ROOT / Path(reference['path'].replace('\\', '/'))
    if not raw_path.is_file():
        raise FileNotFoundError(f'Manifest-referenced raw is missing for {symbol}.')
    raw_hash_checks.append(sha256(raw_path) == reference['sha256'])
assert len(raw_hash_checks) == 138 and all(raw_hash_checks)

# The VIX raw has a capture-manifest reference, which must match now.
vix_raw_hash = sha256(VIX_RAW_PATH)
assert vix_raw_hash == vix_capture_manifest['raw_sha256']
assert vix_raw_hash == alignment_manifest['inputs']['vix_raw']['sha256_capture_manifest_reference']
assert vix_raw_hash == alignment_manifest['inputs']['vix_raw']['sha256_verified_now']

# Alignment outputs are checked against hashes created in that alignment run.
for output_name, reference in alignment_manifest['outputs'].items():
    output_path = ROOT / Path(reference['path'])
    assert output_path.is_file() and sha256(output_path) == reference['sha256'], output_name

# These two Parquet hashes were calculated during the alignment run, not supplied as historical references by the sealed run manifest.
price_hash_matches_alignment_run = sha256(PRICE_MATRIX_PATH) == alignment_manifest['inputs']['price_calendar']['sha256_computed_now']
return_hash_matches_alignment_run = sha256(RETURN_MATRIX_PATH) == alignment_manifest['inputs']['simple_return_matrix']['sha256_computed_now']
assert price_hash_matches_alignment_run and return_hash_matches_alignment_run
assert not alignment_manifest['inputs']['price_calendar']['historical_reference_hash_available']
assert not alignment_manifest['inputs']['simple_return_matrix']['historical_reference_hash_available']

eligible_catalog = pl.read_csv(ELIGIBLE_CATALOG_PATH).filter(pl.col('universe_status') == 'ELIGIBLE')
eligible_symbols = eligible_catalog['symbol'].to_list()
assert len(eligible_symbols) == 125 and len(set(eligible_symbols)) == 125

all_gold_prices = pl.read_parquet(PRICE_MATRIX_PATH)
return_matrix = pl.read_parquet(RETURN_MATRIX_PATH)
assert all_gold_prices.columns[0] == return_matrix.columns[0] == 'session_date'
return_symbols = return_matrix.columns[1:]
assert return_symbols == eligible_symbols
eligible_prices = all_gold_prices.select(['session_date', *return_symbols])
assert eligible_prices.shape == return_matrix.shape == (2205, 126)

published_calendar_dates = [stamp.date() for stamp in xcals.get_calendar('XNYS').sessions if start <= stamp.date() <= end]
for matrix in (eligible_prices, return_matrix):
    dates = matrix['session_date'].to_list()
    assert dates == published_calendar_dates
    assert dates == sorted(dates) and len(set(dates)) == len(dates) == 2205

first_return_row = return_matrix.slice(0, 1).select(pl.exclude('session_date'))
subsequent_return_rows = return_matrix.slice(1).select(pl.exclude('session_date'))
assert all(value == 1 for value in first_return_row.null_count().row(0))
assert all(value == 0 for value in subsequent_return_rows.null_count().row(0))
assert not any(subsequent_return_rows.select(pl.all().is_nan().any()).row(0))
assert not any(subsequent_return_rows.select(pl.all().is_infinite().any()).row(0))

# A original: same usable rows and component-wise log transformation as notebook 02.
usable_returns = return_matrix.slice(1).select(['session_date', *return_symbols])
historical_dates_original = np.asarray(usable_returns['session_date'].to_list())
simple_return_values_original = usable_returns.select(pl.exclude('session_date')).to_numpy()
if np.any(simple_return_values_original <= -1.0):
    raise ValueError('A simple return is outside the log-return domain.')
A_log_returns_original = np.log1p(simple_return_values_original)
assert A_log_returns_original.shape == (2204, 125) and np.isfinite(A_log_returns_original).all()
assert historical_dates_original.tolist() == published_calendar_dates[1:]

# Full audit table is retained; only ALIGNED rows form the working subset.
alignment_audit = pl.read_parquet(ALIGNMENT_TABLE_PATH).sort('return_row_id')
expected_audit_columns = ['return_row_id', 'return_date', 'return_start_date', 'vix_source_date', 'vix_close', 'alignment_status']
assert alignment_audit.columns == expected_audit_columns and alignment_audit.shape == (2204, 6)
assert alignment_audit['return_row_id'].to_list() == list(range(2204))
assert alignment_audit['return_date'].to_list() == historical_dates_original.tolist()
assert alignment_audit['return_date'].is_sorted() and alignment_audit['return_date'].n_unique() == 2204

alignment_mask = (alignment_audit['alignment_status'] == 'ALIGNED').to_numpy()
aligned_source_row_ids = alignment_audit.filter(pl.col('alignment_status') == 'ALIGNED')['return_row_id'].to_numpy()
A_log_returns_aligned = A_log_returns_original[alignment_mask, :]
aligned_return_dates = historical_dates_original[alignment_mask]
vix_associated = alignment_audit.filter(pl.col('alignment_status') == 'ALIGNED')['vix_close'].to_numpy()
assert A_log_returns_aligned.shape == (2203, 125)
assert aligned_return_dates.shape == vix_associated.shape == aligned_source_row_ids.shape == (2203,)
assert np.array_equal(A_log_returns_aligned, A_log_returns_original[aligned_source_row_ids, :])
assert np.isfinite(A_log_returns_aligned).all() and np.isfinite(vix_associated).all() and np.all(vix_associated > 0)

vix_payload = json.loads(VIX_RAW_PATH.read_text(encoding='utf-8'))
assert vix_payload.get('symbol') == 'VIX' and vix_payload.get('interval') == 'daily'
vix_by_date = {date.fromisoformat(str(item['date'])): float(item['close']) for item in vix_payload['data']}
aligned_audit = alignment_audit.filter(pl.col('alignment_status') == 'ALIGNED')
alignment_controls = {
    'Entradas con hashes de referencia disponibles verificados': all(raw_hash_checks) and vix_raw_hash == vix_capture_manifest['raw_sha256'],
    'Hashes de Parquet coinciden con cálculo de la alineación (no referencia histórica)': price_hash_matches_alignment_run and return_hash_matches_alignment_run,
    'A original tiene 2,204 × 125 y fechas ordenadas': A_log_returns_original.shape == (2204, 125) and historical_dates_original.tolist() == sorted(historical_dates_original.tolist()),
    'Auditoría coincide exactamente con fechas de A': alignment_audit['return_date'].to_list() == historical_dates_original.tolist(),
    'Orden de los 125 símbolos coincide con el catálogo elegible': return_symbols == eligible_symbols,
    'Subconjunto conserva exactamente las filas originales seleccionadas': np.array_equal(A_log_returns_aligned, A_log_returns_original[aligned_source_row_ids, :]),
    'Retornos y VIX alineados son finitos; VIX positivo': bool(np.isfinite(A_log_returns_aligned).all() and np.isfinite(vix_associated).all() and (vix_associated > 0).all()),
    'Relaciones x-1 y x-2 usan el calendario de acciones': all(row['return_start_date'] == published_calendar_dates[int(row['return_row_id'])] and row['vix_source_date'] == published_calendar_dates[int(row['return_row_id']) - 1] for row in aligned_audit.to_dicts()),
    'VIX es anterior a inicio y fecha del retorno': all(row['vix_source_date'] < row['return_start_date'] < row['return_date'] for row in aligned_audit.to_dicts()),
    'VIX coincide con raw en la fecha fuente exacta': all(vix_by_date[row['vix_source_date']] == row['vix_close'] for row in aligned_audit.to_dicts()),
    'Una sola fila no alineable: 2018-01-03': alignment_audit.filter(pl.col('alignment_status') != 'ALIGNED').to_dicts() == [{'return_row_id': 0, 'return_date': date(2018, 1, 3), 'return_start_date': date(2018, 1, 2), 'vix_source_date': None, 'vix_close': None, 'alignment_status': 'UNALIGNABLE_PRE_PUBLISHED_CALENDAR_START'}],
    'No se usa VIX posterior al corte ni fechas exclusivas de VIX': all(row['vix_source_date'] <= end for row in aligned_audit.to_dicts()) and set(aligned_audit['vix_source_date'].to_list()).issubset(set(published_calendar_dates)),
}
if not all(alignment_controls.values()):
    raise AssertionError([name for name, passed in alignment_controls.items() if not passed])


## Entradas preparadas

La salida siguiente es una auditoría compacta. `A_log_returns_original` conserva las 2,204 filas originales; `A_log_returns_aligned`, `vix_associated`, `aligned_return_dates` y `aligned_source_row_ids` contienen únicamente las 2,203 filas explícitamente alineadas. No se han creado probabilidades condicionadas, estados discretos, umbrales ni simulaciones.

In [3]:
from IPython.display import display

summary = pl.DataFrame({
    'item': ['Corte publicado', 'Calendario completo de precios', 'A original de log-retornos', 'Auditoría de alineación', 'Subconjunto ALIGNED', 'Instrumentos'],
    'value': ['2018-01-02 a 2026-10-09', '2,205 sesiones', '2,204 × 125', '2,204 filas', '2,203 × 125', '125'],
    'convention': ['Retorno x: cierre x-1 a cierre x', 'Incluye la fecha inicial necesaria', 'log1p de retornos simples utilizables', 'VIX en sesión de acciones x-2', 'Sin imputación ni reordenamiento', 'Orden conservado'],
})

example_symbols = ['SPY', 'TLT', 'GLD']
example_indices = [return_symbols.index(symbol) for symbol in example_symbols]
aligned_examples = aligned_audit.select(['return_row_id', 'return_date', 'return_start_date', 'vix_source_date', 'vix_close']).with_columns(
    *[pl.Series(symbol, A_log_returns_aligned[:, index]) for symbol, index in zip(example_symbols, example_indices)]
)
excluded_rows = alignment_audit.filter(pl.col('alignment_status') != 'ALIGNED')
control_table = pl.DataFrame({'control': list(alignment_controls), 'result': ['PASS' if passed else 'FAIL' for passed in alignment_controls.values()]})

display(summary)
display(aligned_examples.head(5))
display(aligned_examples.tail(5))
display(excluded_rows)
display(control_table)


item,value,convention
str,str,str
"""Corte publicado""","""2018-01-02 a 2026-10-09""","""Retorno x: cierre x-1 a cierre…"
"""Calendario completo de precios""","""2,205 sesiones""","""Incluye la fecha inicial neces…"
"""A original de log-retornos""","""2,204 × 125""","""log1p de retornos simples util…"
"""Auditoría de alineación""","""2,204 filas""","""VIX en sesión de acciones x-2"""
"""Subconjunto ALIGNED""","""2,203 × 125""","""Sin imputación ni reordenamien…"
"""Instrumentos""","""125""","""Orden conservado"""


return_row_id,return_date,return_start_date,vix_source_date,vix_close,SPY,TLT,GLD
i64,date,date,date,f64,f64,f64,f64
1,2018-01-04,2018-01-03,2018-01-02,9.77,0.004206,-0.000159,0.005114
2,2018-01-05,2018-01-04,2018-01-03,9.15,0.006642,-0.00286,-0.001037
3,2018-01-08,2018-01-05,2018-01-04,9.22,0.001827,-0.000637,-0.00016
4,2018-01-09,2018-01-08,2018-01-05,9.22,0.002261,-0.013463,-0.004639
5,2018-01-10,2018-01-09,2018-01-08,9.52,-0.001531,-0.001211,0.002402


return_row_id,return_date,return_start_date,vix_source_date,vix_close,SPY,TLT,GLD
i64,date,date,date,f64,f64,f64,f64
2199,2026-10-05,2026-10-02,2026-10-01,16.39,0.006721,-0.004787,-0.001553
2200,2026-10-06,2026-10-05,2026-10-02,15.31,0.005483,0.002202,0.007141
2201,2026-10-07,2026-10-06,2026-10-05,15.52,-0.002403,-0.001748,-0.016857
2202,2026-10-08,2026-10-07,2026-10-06,15.01,-0.004242,0.009354,0.007263
2203,2026-10-09,2026-10-08,2026-10-07,15.08,0.005977,0.001412,0.015619


return_row_id,return_date,return_start_date,vix_source_date,vix_close,alignment_status
i64,date,date,date,f64,str
0,2018-01-03,2018-01-02,null,null,"""UNALIGNABLE_PRE_PUBLISHED_CALE…"


control,result
str,str
"""Entradas con hashes de referen…","""PASS"""
"""Hashes de Parquet coinciden co…","""PASS"""
"""A original tiene 2,204 × 125 y…","""PASS"""
"""Auditoría coincide exactamente…","""PASS"""
"""Orden de los 125 símbolos coin…","""PASS"""
…,…
"""Relaciones x-1 y x-2 usan el c…","""PASS"""
"""VIX es anterior a inicio y fec…","""PASS"""
"""VIX coincide con raw en la fec…","""PASS"""


## 2. ¿Qué estado del mercado queremos representar?

Denotemos por $z_t$ el nivel previo del VIX asociado a la observación histórica $t$. Después de aplicar la convención temporal auditada, tenemos 2,203 pares utilizables: cada fila de log-retornos conserva su vector completo de 125 instrumentos y su $z_t$ asociado.

En cambio, $z^*$ es un **estado objetivo**: la condición para la que, más adelante, podríamos construir una distribución de escenarios. Por ejemplo, si planteáramos hipotéticamente $z^*=30$, la pregunta sería: *queremos investigar una distribución que dé mayor relevancia a observaciones precedidas por condiciones próximas a ese nivel*.

Ese ejemplo no fija un parámetro del modelo y elegir un estado objetivo no equivale a pronosticar que el VIX alcanzará ese nivel. El objetivo podría representar una condición observada en una fecha de referencia o una condición hipotética de estrés. El VIX asociado es una descripción parcial del estado: no caracteriza por completo el mercado.

Todavía no definimos probabilidades ni un criterio de proximidad. En particular, los 125 retornos de cada fila permanecen juntos e intactos; este bloque sólo contextualiza el vector histórico $z_t$.

In [4]:
# La mediana sigue la inversa discreta de la CDF empírica uniforme: q_α = inf{x : F_hat(x) ≥ α}.
# No hay interpolación ni selección de un estado objetivo en este bloque.
def empirical_quantile_inverse_cdf(values: np.ndarray, alpha: float) -> float:
    if not (0 < alpha <= 1):
        raise ValueError('alpha must be in (0, 1].')
    ordered = np.sort(np.asarray(values, dtype=float))
    if ordered.size == 0 or not np.isfinite(ordered).all():
        raise ValueError('values must be non-empty and finite.')
    index = max(0, math.ceil(alpha * ordered.size) - 1)
    return float(ordered[index])

z_historical = vix_associated.copy()
aligned_dates_for_state = aligned_return_dates.copy()
aligned_row_ids_for_state = aligned_source_row_ids.copy()

vix_min = float(np.min(z_historical))
vix_median_discrete = empirical_quantile_inverse_cdf(z_historical, 0.5)
vix_max = float(np.max(z_historical))
last_aligned_pair = aligned_audit.tail(1)

state_context_checks = {
    'El vector z_t tiene exactamente las 2,203 filas alineadas': z_historical.shape == (2203,),
    'z_t coincide exactamente con VIX de la auditoría ALIGNED': np.array_equal(z_historical, aligned_audit['vix_close'].to_numpy()),
    'Fechas e identificadores del contexto coinciden con el subconjunto alineado': np.array_equal(aligned_dates_for_state, aligned_audit['return_date'].to_numpy()) and np.array_equal(aligned_row_ids_for_state, aligned_audit['return_row_id'].to_numpy()),
    'El último par coincide con la última fila de auditoría': (aligned_dates_for_state[-1] == last_aligned_pair['return_date'][0] and z_historical[-1] == last_aligned_pair['vix_close'][0]),
    'El resumen mínimo/mediana/máximo procede de z_t': vix_min <= vix_median_discrete <= vix_max and vix_median_discrete in z_historical,
    'Los objetos del bloque de carga conservan identidad y orden': (A_log_returns_original.shape == (2204, 125) and np.array_equal(A_log_returns_original, np.log1p(simple_return_values_original)) and np.array_equal(alignment_mask, alignment_audit['alignment_status'].to_numpy() == 'ALIGNED') and np.array_equal(A_log_returns_aligned, A_log_returns_original[alignment_mask, :])),
}
if not all(state_context_checks.values()):
    raise AssertionError([name for name, passed in state_context_checks.items() if not passed])

state_context_summary = pl.DataFrame({
    'observaciones_alineadas': [z_historical.size],
    'vix_mínimo': [vix_min],
    'vix_mediana_empírica_discreta': [vix_median_discrete],
    'vix_máximo': [vix_max],
})
last_pair_display = last_aligned_pair.select(['return_date', 'vix_source_date', 'vix_close']).rename({
    'return_date': 'fecha_del_retorno',
    'vix_source_date': 'fecha_de_origen_del_vix',
    'vix_close': 'último_vix_asociado_a_la_muestra_alineada',
})
state_context_control_table = pl.DataFrame({
    'control': list(state_context_checks),
    'result': ['PASS' if passed else 'FAIL' for passed in state_context_checks.values()],
})

display(state_context_summary)
display(last_pair_display)
display(state_context_control_table)


observaciones_alineadas,vix_mínimo,vix_mediana_empírica_discreta,vix_máximo
i64,f64,f64,f64
2203,9.15,17.78,82.69


fecha_del_retorno,fecha_de_origen_del_vix,último_vix_asociado_a_la_muestra_alineada
date,date,f64
2026-10-09,2026-10-07,15.08


control,result
str,str
"""El vector z_t tiene exactament…","""PASS"""
"""z_t coincide exactamente con V…","""PASS"""
"""Fechas e identificadores del c…","""PASS"""
"""El último par coincide con la …","""PASS"""
"""El resumen mínimo/mediana/máxi…","""PASS"""
"""Los objetos del bloque de carg…","""PASS"""


## Preparado, sin resolver todavía

Ya distinguimos los estados históricos del estado objetivo. La siguiente pregunta es cómo traducir ese objetivo en probabilidades sobre las filas históricas.

La muestra sigue alineada, con retornos originales, orden de símbolos conservado y un VIX previo asociado por una convención temporal auditable. Aún no hemos transformado el VIX en estados, umbrales o probabilidades.